# Understanding the Data
After unpacking the 2011-2023 Gwinnett zip file, I saw that there were actual a number of the excel spreadsheets which did contain sales information.

The format of this information is a little interesting, because there's several columns that are used to identify:
- LRSNum
- PIN
- LOCADDR
- LocCity
- LocZip
- LEGALAC
- PCDESC
- ZONEDESC

Then some attributes that are used to identify a specific owner, in this context a grantee.
- OWNER1
- OWNER2
- MAILADDR
- MAILCITY
- MAILSTAT
- Sale Date (Broken into SALE1D, SALE2D, SALE3D for the last transcations)
- Sale Amount (Broken into SALE1AMT, SALE2AMT, SALE3AMT)
- Grantor (Broken into GRANTOR1, GRANTOR2, GRANTOR3)
- Document Reference (Broken into DOC1REF, DOC2REF, DOC3REF) - I'm assuming something like the page and book numbers in other counties

The creation of the sales records is interesting because OWNER1 is the GRANTEE of GRANTOR1, GRANTOR1 is the GRANTEE of GRANTOR2, etc.  
So only some of the records will have OWNER1, OWNER2, MAILADDR, MAILCITY, MAILSTAT, specifically those records corresponding to the final sale of the contemporaneous owner (at time of recording).

**Road Map**  
1. Pick out a singular tax assessment document per year. There seems to be a lot of duplicate versions, with largely the same information. I'm just picking out the sheet that has the most rows / the latest sales date, indicating it's the most updated.
2. Check to ensure that the important columns are present across all of the assessment documents.
3. Duplicate the relevant property-wise columns for each of the sales transactions, to produce a unique row for each sale.
4. Merge all the years sales data.
5. Sort by those rows which have non-empty MAILADDR columns.
6. Deduplicate, taking the first value, ensuring that we take the most up to date sales transactions (those with MAILADDR), if possible

**Selected Tax Assessment Documents**  
Current Ownership_2011 Digest Assessed Values.xlsx  
2012 Gwinnett Digest TAFull_Ownership_CD7.xlsx  
2013 Tax Digest Ownership_CD7.xlsx  
2014 Property Ownership CD7.xlsx  
2015 Property Ownership CD7.xlsx  
2016 Property Ownership CD7.xlsx  
2017 Property Ownership CD7.xlsx  
2018 Property Ownership  CD7.xlsx  
2019 Property Ownership CD7.xlsx  
2020 Property Ownership CD7.xlsx  
2021 Property Ownership CD7.xlsx  
2022 Property Ownership CD7.xlsx  
2023 Property Ownership CD7.xlsx  

In [1]:
import pandas as pd
import os
from datetime import datetime

DATA_PATH = "../../data/gwinnett"
OUT_PATH = "../../data/gwinnett/out"

I have renamed all of hte files to follow the 20{XX} Property Ownership CD7.xlsx format, for convenience.

In [21]:
year_dfs = []

for file_p in os.listdir(DATA_PATH):
    if file_p.endswith(".xlsx"):
        year = int(file_p.split(" ")[0])
        df = pd.read_excel(os.path.join(DATA_PATH, file_p))

        year_dfs.append((year, df))

In [22]:
year_dfs.sort(key = lambda x : x[0])

In [23]:
for year, df in year_dfs:
    print(year)
    print(df.columns)

2011
Index(['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP', 'LEGALAC', 'PCDESC', 'ZONEDESC', 'EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1', 'SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 'DOC3REF'],
      dtype='object')
2012
Index(['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP', 'LEGALAC', 'PCDESC', 'ZONEDESC', 'EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1', 'SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 

In [37]:
for year, df in year_dfs:
    print(year)
    print(set(df.columns) - set(year_dfs[-1][1].columns))

2011
{'MAILCITY', 'MAILADDR', 'MAILZIP', 'EXEMPT1', 'GRANTOR3', 'GRANTOR2', 'LocCity', 'LocState', 'LOCADDR', 'MAILSTAT', 'OWNER2', 'EXEMPT1D', 'OWNER1', 'LocZip', 'GRANTOR1'}
2012
{'MAILCITY', 'MAILADDR', 'MAILZIP', 'EXEMPT1', 'GRANTOR3', 'GRANTOR2', 'LocCity', 'LocState', 'LOCADDR', 'MAILSTAT', 'OWNER2', 'EXEMPT1D', 'OWNER1', 'LocZip', 'GRANTOR1'}
2013
{'yrbuilt', 'MAILADDR', 'MAILZIP', 'EXEMPT1', 'GRANTOR3', 'GRANTOR2', 'PROPERTY CITY', 'MAILSTAT', 'OWNER2', 'PROPERTY ADDR', 'stories', 'PROPERTY ZIP', 'FinSize', 'OWNER1', 'MAILCITY', 'GRANTOR1'}
2014
{'EXEMPT1D', 'EXEMPT1'}
2015
set()
2016
set()
2017
set()
2018
set()
2019
set()
2020
set()
2021
set()
2022
set()
2023
set()


It looks like 2011 and 2012 have their own format, 2013 has its own format, and the rest of the years are the same. So this is how I will be parsing them.

In [ ]:
# 2013 has an abnormal page structure
year_dfs[2] = (2013, pd.read_excel(os.path.join(DATA_PATH, "2013 Property Ownership CD7.xlsx"), sheet_name="real_master_0001"))

In [94]:
property_cols = ['LRSNum', 'PIN', 'Public_NeiNum', 'LOCADDR', 'LocCity', 'LocState',
       'LocZip', 'LEGALAC', 'PCDESC', 'ZONEDESC']

owner_cols = ['OWNER1', 'OWNER2', 'MAILADDR', 'MAILCITY', 'MAILSTAT',
       'MAILZIP']

drop_cols = ['EXEMPT1', 'EXEMPT1D',
       'ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1',
       'TAXDWLG1', 'TAXOTH1', 'TAXTOT1']

sale_cols = ['SALE1D', 'SALE2D', 'SALE3D',
       'SALE1AMT', 'SALE2AMT', 'SALE3AMT', 'GRANTOR1', 'GRANTOR2', 'GRANTOR3',
       'DOC1REF', 'DOC2REF', 'DOC3REF']

sales_2011_2013 = []
for i in range(2):
    year, year_df = year_dfs[i]
    year_df['SALE1D'] = pd.to_datetime(year_df['SALE1D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE2D'] = pd.to_datetime(year_df['SALE2D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE3D'] = pd.to_datetime(year_df['SALE3D'], format="%m/%d/%Y", errors="coerce")
    year_df['YEAR_RECORDED'] = year
    grantor3_sales = year_df.loc[~year_df['SALE3D'].isna(), :].copy()
    grantor2_sales = year_df.loc[~year_df['SALE2D'].isna(), :].copy()
    grantor1_sales = year_df.loc[~year_df['SALE1D'].isna(), :].copy()
    
    grantor3_sales['GRANTOR'] = grantor3_sales['GRANTOR3']
    grantor3_sales['GRANTEE'] = grantor3_sales['GRANTOR2']
    grantor3_sales['SALEDT'] = grantor3_sales['SALE3D']
    grantor3_sales['SALEAMT'] = grantor3_sales['SALE3AMT']
    grantor3_sales['DOCREF'] = grantor3_sales['DOC3REF']
    grantor3_sales.loc[:, owner_cols] = pd.NA
    grantor3_sales = grantor3_sales.drop(columns=drop_cols + sale_cols)

    grantor2_sales['GRANTOR'] = grantor2_sales['GRANTOR2']
    grantor2_sales['GRANTEE'] = grantor2_sales['GRANTOR1']
    grantor2_sales['SALEDT'] = grantor2_sales['SALE2D']
    grantor2_sales['SALEAMT'] = grantor2_sales['SALE2AMT']
    grantor2_sales['DOCREF'] = grantor2_sales['DOC2REF']
    grantor2_sales.loc[:, owner_cols] = pd.NA
    grantor2_sales = grantor2_sales.drop(columns=drop_cols + sale_cols)

    grantor1_sales['GRANTOR'] = grantor1_sales['GRANTOR1']
    grantor1_sales['GRANTEE'] = grantor1_sales['OWNER1']
    grantor1_sales['SALEDT'] = grantor1_sales['SALE1D']
    grantor1_sales['SALEAMT'] = grantor1_sales['SALE1AMT']
    grantor1_sales['DOCREF'] = grantor1_sales['DOC1REF']
    # Notably, no clearing of the owner1 columns this time because they actually exist
    grantor1_sales = grantor1_sales.drop(columns=drop_cols + sale_cols)

    sales_2011_2013.append(grantor1_sales)
    sales_2011_2013.append(grantor2_sales)
    sales_2011_2013.append(grantor3_sales)

sales_2011_2013_df = pd.concat(sales_2011_2013)

In [95]:
# Normalize with the other dataframes
sales_2011_2013_df = sales_2011_2013_df.rename(columns={'LOCADDR': 'PROPERTYSTREET', 'LocCity': 'PROPERTYCity',
                                                        'LocState': 'PROPERTYState', 'LocZip': 'PROPERTYZip',
                                                        'OWNER1': 'OWNERNAME1', 'OWNER2': 'OWNERNAME2',
                                                        'MAILADDR': 'OWNERADDRESS1', 'MAILCITY': "OWNERCITY",
                                                        "MAILSTAT": "OWNERSTATE", "MAILZIP": "OWNERZIP"})

In [96]:
property_cols = ['LRSNum', 'PIN', 'Public_NeiNum', 'PROPERTYSTREET', 'PROPERTYCity',
       'PROPERTYState', 'PROPERTYZip', 'LEGALAC', 'LEGAL1', 'PCDESC', 'ZONEDESC']

owner_cols = ['OWNERNAME1', 'OWNERNAME2',
       'OWNERADDRESS1', 'OWNERCITY', 'OWNERSTATE', 'OWNERZIP']

drop_cols = ['ASSMNT1D', 'LANDVAL1', 'DWLGVAL1', 'OTHVAL1', 'TOTVAL1', 'TAXLAND1', 'TAXDWLG1',
       'TAXOTH1', 'TAXTOT1']

sale_cols = ['SALE1D', 'SALE2D', 'SALE3D', 'SALE1AMT',
       'SALE2AMT', 'SALE3AMT', 'GRANTORNAME1', 'GRANTORNAME2', 'GRANTORNAME3',
       'DOC1REF', 'DOC2REF', 'DOC3REF']

sales_2014_2023 = []
for i in range(3, len(year_dfs)):
    year, year_df = year_dfs[i]
    year_df['SALE1D'] = pd.to_datetime(year_df['SALE1D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE2D'] = pd.to_datetime(year_df['SALE2D'], format="%m/%d/%Y", errors="coerce")
    year_df['SALE3D'] = pd.to_datetime(year_df['SALE3D'], format="%m/%d/%Y", errors="coerce")
    year_df['YEAR_RECORDED'] = year
    grantor3_sales = year_df.loc[~year_df['SALE3D'].isna(), :].copy()
    grantor2_sales = year_df.loc[~year_df['SALE2D'].isna(), :].copy()
    grantor1_sales = year_df.loc[~year_df['SALE1D'].isna(), :].copy()
    
    grantor3_sales['GRANTOR'] = grantor3_sales['GRANTORNAME3']
    grantor3_sales['GRANTEE'] = grantor3_sales['GRANTORNAME2']
    grantor3_sales['SALEDT'] = grantor3_sales['SALE3D']
    grantor3_sales['SALEAMT'] = grantor3_sales['SALE3AMT']
    grantor3_sales['DOCREF'] = grantor3_sales['DOC3REF']
    grantor3_sales.loc[:, owner_cols] = pd.NA
    grantor3_sales = grantor3_sales.drop(columns=drop_cols + sale_cols)

    grantor2_sales['GRANTOR'] = grantor2_sales['GRANTORNAME2']
    grantor2_sales['GRANTEE'] = grantor2_sales['GRANTORNAME1']
    grantor2_sales['SALEDT'] = grantor2_sales['SALE2D']
    grantor2_sales['SALEAMT'] = grantor2_sales['SALE2AMT']
    grantor2_sales['DOCREF'] = grantor2_sales['DOC2REF']
    grantor2_sales.loc[:, owner_cols] = pd.NA
    grantor2_sales = grantor2_sales.drop(columns=drop_cols + sale_cols)

    grantor1_sales['GRANTOR'] = grantor1_sales['GRANTORNAME1']
    grantor1_sales['GRANTEE'] = grantor1_sales['OWNERNAME1']
    grantor1_sales['SALEDT'] = grantor1_sales['SALE1D']
    grantor1_sales['SALEAMT'] = grantor1_sales['SALE1AMT']
    grantor1_sales['DOCREF'] = grantor1_sales['DOC1REF']
    # Notably, no clearing of the owner1 columns this time because they actually exist
    grantor1_sales = grantor1_sales.drop(columns=drop_cols + sale_cols)

    sales_2014_2023.append(grantor1_sales)
    sales_2014_2023.append(grantor2_sales)
    sales_2014_2023.append(grantor3_sales)

sales_2014_2023_df = pd.concat(sales_2014_2023)
sales_2014_2023_df = sales_2014_2023_df.drop(columns=['EXEMPT1', 'EXEMPT1D'])

In [97]:
total_sales_digest = pd.concat([sales_2011_2013_df, sales_2014_2023_df])

In [106]:
total_sales_digest['HASOWNER'] = ~total_sales_digest['OWNERNAME1'].isna()
total_sales_digest['PIN'] = total_sales_digest['PIN'].str.strip()

In [99]:
total_sales_digest = total_sales_digest.sort_values(by="HASOWNER", ascending=False)

In [107]:
original_rows = total_sales_digest.shape[0]

total_sales_digest_dedup = total_sales_digest.drop_duplicates(subset=['LRSNum', 'PIN', 'SALEDT'])
new_rows = total_sales_digest_dedup.shape[0]

print(f"Rows Removed: {original_rows - new_rows}, {new_rows} Remaining")

Rows Remove: 7552224, 936953 Remaining


This passes the sniff test, since the total number of 2023 rows was around 300000, so assuming that there were some properties which were sold more than 3 times, wherein the 2012 file contributed different sales records than 2023 for example, then this checks out.

In [108]:
total_sales_digest_dedup = total_sales_digest_dedup.sort_values(by="SALEDT", ascending=True)

In [109]:
total_sales_digest_dedup.to_csv(os.path.join(OUT_PATH, "GWINNET_SALES_RAW.csv"), index=False)

# Merging with Tax Digest
The sales digest in this scenario is actually derived from the tax digest, so it is a little redundant, but I guess it's a reverse mapping if you need that.

In [6]:
TAX_DIGEST_PATH = "/Users/tpeng/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Housing and Urban Policy (HUP) Lab - Documents/Data Engineering/Output/Most up-to date/Gwinnett/3-ownership_keys/gwinnett_digest_full_final.csv"
SALES_DIGEST_PATH = os.path.join(OUT_PATH, "GWINNET_SALES_FINAL.csv")

print(f"Correct Tax Digest Path {os.path.exists(TAX_DIGEST_PATH)}")
print(f"Correct Sales Digest Path {os.path.exists(SALES_DIGEST_PATH)}")

Correct Tax Digest Path True
Correct Sales Digest Path True


In [8]:
tax_digest = pd.read_csv(TAX_DIGEST_PATH)
sales_digest = pd.read_csv(SALES_DIGEST_PATH)

/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_51024/4216750817.py:1: DtypeWarning: Columns (7,19,28,29,30,40,42,43,44,49) have mixed types. Specify dtype option on import or set low_memory=False.
  tax_digest = pd.read_csv(TAX_DIGEST_PATH)


In [9]:
display(tax_digest.dtypes)
display(sales_digest.dtypes)

tax_year                 int64
lrs_num                  int64
pin                     object
public_nei_num           int64
propertystreet          object
property_city           object
property_state          object
property_zip            object
ownername1              object
ownername2              object
owneraddress1           object
ownercity               object
ownerstate              object
ownerzip                object
legalac                float64
pcdesc                  object
zonedesc                object
exempt1                float64
exempt1d               float64
assmnt1d                object
landval1               float64
dwlgval1               float64
othval1                float64
totval1                float64
taxland1               float64
taxdwlg1               float64
taxoth1                float64
taxtot1                float64
sale1d                  object
sale2d                  object
sale3d                  object
sale1amt               float64
sale2amt

LRSNum              int64
PIN                object
Public_NeiNum       int64
PROPERTYSTREET     object
PROPERTYCity       object
PROPERTYState      object
PROPERTYZip        object
OWNERNAME1         object
OWNERNAME2         object
OWNERADDRESS1      object
OWNERCITY          object
OWNERSTATE         object
OWNERZIP           object
LEGALAC           float64
PCDESC             object
ZONEDESC           object
YEAR_RECORDED       int64
GRANTOR            object
GRANTEE            object
SALEDT             object
SALEAMT           float64
DOCREF             object
LEGAL1             object
DISTNUM            object
DISTNUM_DESC       object
PROPCLAS          float64
PROPCLAS_DESC      object
HASOWNER             bool
dtype: object

In [ ]:
sales_digest['SALE_YR'] = pd.to_datetime(sales_digest['SALEDT']).dt.year

In [42]:
tax_digest['tax_year'].value_counts()

tax_year
2023    302177
2022    299951
2021    295120
2020    290065
2019    288376
2018    285589
2017    281418
2016    279608
2015    277690
2014    275965
2013    274966
2012    274574
2011    274313
Name: count, dtype: int64

In [43]:
merge_cols = ['tax_year', 'lrs_num', 'exempt1', 'exempt1d', 'assmnt1d',
       'landval1', 'dwlgval1', 'othval1', 'totval1', 'taxland1', 'taxdwlg1',
       'taxoth1', 'taxtot1', 'mod_own_adrstr', 'mod_unitno',
       'mod_own_adrsuf2', 'mod_own_adrsuf', 'mod_ownerzip', 'owner_addr',
       'own_corp_flag', 'rental_flag', 'mod_owneraddress1',
       'mod_owneraddress1_B']

tax_subset = tax_digest[merge_cols]
sales_subset = sales_digest[sales_digest['SALE_YR'] >= 2011]
sales_tax_merged = pd.merge(sales_subset, tax_subset, how="left", left_on=["LRSNum", "SALE_YR"], right_on=["lrs_num", "tax_year"])

total_rows = sales_subset.shape[0]
merged_rows = sales_tax_merged['lrs_num'].notna().sum()

print(f"{merged_rows} out of {total_rows} matched: {merged_rows / total_rows * 100}%")

390804 out of 398852 matched: 97.98220893965681%


In [45]:
sales_tax_merged[sales_tax_merged['lrs_num'].isna()].head(10)

,LRSNum,PIN,Public_NeiNum,PROPERTYSTREET,PROPERTYCity,PROPERTYState,PROPERTYZip,OWNERNAME1,OWNERNAME2,OWNERADDRESS1,...,mod_own_adrstr,mod_unitno,mod_own_adrsuf2,mod_own_adrsuf,mod_ownerzip,owner_addr,own_corp_flag,rental_flag,mod_owneraddress1,mod_owneraddress1_B
2615,33271689,R5134 253,5639,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3475,33353538,R7232 407,7895,775 LAURA JEAN CT,BUFORD,NaN,30518,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10690,33271689,R5134 253,5639,2431 ALEXANDER TOP PLACE,GRAYSON,NaN,NaN,NORRIS-CONSTABLE GLENDA-MARIE P,CONSTABLE ALBERT A,PO BOX 1943,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13508,33363396,R7138 448,7099,3165 MORGAN RD,BUFORD,NaN,30519,LIU TINGTING,NaN,3165 MORGAN RD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30560,2811641,R7011 134,9061,175 PARK ACCESS DR,LAWRENCEVILLE,NaN,30045,STOVALL PROPERTIES INC,NaN,1000 PEACHTREE INDUSTRIAL BLVD STE 6-198,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30578,33253877,R7082 303,8034,1916 LEBANON RD,LAWRENCEVILLE,NaN,30043,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30588,33253875,R7082 301,8074,1896 LEBANON RD,LAWRENCEVILLE,NaN,30043,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30594,33253876,R7082 302,8074,1906 LEBANON RD,LAWRENCEVILLE,NaN,30043,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30611,33243056,R7285 186,7312,768 MIDDLE FORK TRL,SUWANEE,,30024,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30649,829579,R6129 011,6010,SIR GREGORY MANOR,LAWRENCEVILLE,NaN,30044,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
sales_tax_merged[sales_tax_merged['lrs_num'].isna()]['SALE_YR'].value_counts()

SALE_YR
2020    2160
2017    1085
2021    1050
2019     889
2022     816
2016     714
2018     611
2015     404
2014     229
2013      88
2012      52
2011       4
2102       1
2111       1
Name: count, dtype: int64

In [46]:
sales_tax_merged.to_csv(os.path.join(OUT_PATH, "GWINNET_SALES_TAX_DIGEST_FINAL.csv"), index=False)